<a href="https://colab.research.google.com/github/kimkihyun1/ssafy-ai-challenge-vqa/blob/main/vqa_qwen35_27b_A100_FULL5073_TTA4_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SSAFY VQA — Qwen3.5-27B / A100 80GB / FULL 5,073 최종 제출

validation 학습을 제거하고 **전체 train 5,073개를 한 번만 학습**하는 최종 실행용입니다.

```text
Qwen/Qwen3.5-27B
        ↓
전체 train 5,073
        ↓
fresh LoRA / 1 epoch
        ↓
BF16 우선
  └ OOM → 8-bit 자동 fallback
        ↓
test 5,074
        ↓
TTA=4
        ↓
submission.csv
```

유지하는 핵심:
- AnswerOnly LM loss
- 학습 중 선택지 random permutation
- `a/b/c/d` next-token scoring
- test TTA=4
- adapter / TTA cache / submission을 Drive에 저장
- TTA 중 런타임이 끊겨도 완료된 permutation 재사용

기본값:
```python
MODEL_PRECISION = "auto"
FULL_EPOCHS = 1
FULL_LR = 1e-4
MAX_VISUAL_TOKENS = 1024
TTA_PERMS = 4
```

## 0. 패키지 설치

이 셀을 한 번 실행한 뒤 **런타임 → 세션 다시 시작**을 하세요.  
재시작 후에는 이 설치 셀을 건너뛰고 Drive mount부터 순서대로 실행합니다.

In [ ]:
!pip uninstall -y torchao
!pip install -q -U git+https://github.com/huggingface/transformers
!pip install -q -U \
    "peft>=0.20.0" \
    "accelerate>=1.8.0" \
    "bitsandbytes>=0.46.1" \
    "pandas==2.2.3" \
    "scikit-learn>=1.5" \
    pillow tqdm safetensors

print("설치 완료 ✅")
print('이 셀을 처음 실행했다면 "런타임 > 세션 다시 시작" 후 Drive mount부터 실행하세요.')

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 95.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 136.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 122.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 9.5 MB/s eta 0:00:00
설치 완료 ✅
이 셀을 처음 실행했다면 "런타임 > 세션 다시 시작" 후 Drive mount부터 실행하세요.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 1. 설정

In [ ]:
# ============================================================
# 최종 실행 설정
# ============================================================

ROOT = "/content/drive/MyDrive/ssafy-16-1-ai-8-28"

MODEL_ID = "Qwen/Qwen3.5-27B"
SEED = 42

# "auto" : BF16 preflight -> OOM 시 8-bit 자동 fallback
# "bf16" : BF16만
# "8bit" : 처음부터 8-bit
MODEL_PRECISION = "auto"

FULL_EPOCHS = 1
FULL_LR = 1e-4

MIN_VISUAL_TOKENS = 256
MAX_VISUAL_TOKENS = 1024
# 8-bit에서도 OOM이면 768로 낮추고 런타임 재시작

TRAIN_BATCH = 1
GRAD_ACCUM = 8
INFER_BATCH = 1
NUM_WORKERS = 0

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

TTA_PERMS = 4
COPY_IMAGES_TO_LOCAL = True

print("MODEL_ID:", MODEL_ID)
print("MODEL_PRECISION:", MODEL_PRECISION)
print("FULL_EPOCHS:", FULL_EPOCHS)
print("FULL_LR:", FULL_LR)
print("visual tokens:", MIN_VISUAL_TOKENS, "~", MAX_VISUAL_TOKENS)
print("TTA_PERMS:", TTA_PERMS)

MODEL_ID: Qwen/Qwen3.5-27B
MODEL_PRECISION: auto
FULL_EPOCHS: 1
FULL_LR: 0.0001
visual tokens: 256 ~ 1024
TTA_PERMS: 4


## 2. 환경 / 데이터 로드

In [ ]:
import os, gc, math, random, shutil, json, time
from pathlib import Path
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import transformers
import peft

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
    get_peft_model_state_dict,
    set_peft_model_state_dict,
)

Image.MAX_IMAGE_PIXELS = None
os.environ["TOKENIZERS_PARALLELISM"] = "false"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

assert torch.cuda.is_available(), "GPU 런타임으로 변경하세요."

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16_OK = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16_OK else torch.float16

print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT        :", peft.__version__)
print("GPU         :", GPU_NAME)
print(f"VRAM        : {GPU_GB:.1f} GB")
print("dtype       :", DTYPE)

if GPU_GB < 60:
    print("⚠️ 이 노트북은 A100 80GB 기준입니다. 60GB 미만이면 8-bit 또는 더 낮은 visual token이 필요합니다.")

ROOT = Path(ROOT)

TRAIN_CSV = ROOT / "train.csv"
TEST_CSV = ROOT / "test.csv"
DEV_CSV = ROOT / "dev.csv"
SAMPLE_CSV = ROOT / "sample_submission.csv"

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
dev_df = pd.read_csv(DEV_CSV) if DEV_CSV.exists() else None
sample_df = pd.read_csv(SAMPLE_CSV) if SAMPLE_CSV.exists() else None

CHOICES = ["a", "b", "c", "d"]

required_train = {"id", "path", "question", "a", "b", "c", "d", "answer"}
required_test = {"id", "path", "question", "a", "b", "c", "d"}

assert required_train.issubset(train_df.columns), train_df.columns
assert required_test.issubset(test_df.columns), test_df.columns

train_df["answer"] = train_df["answer"].astype(str).str.strip().str.lower()
train_df = (
    train_df[train_df["answer"].isin(CHOICES)]
    .drop_duplicates("id")
    .reset_index(drop=True)
)
test_df = test_df.drop_duplicates("id").reset_index(drop=True)

print("train:", train_df.shape)
print("test :", test_df.shape)
print("dev  :", None if dev_df is None else dev_df.shape)
print()
print(train_df["answer"].value_counts(normalize=True).sort_index())

PyTorch     : 2.11.0+cu128
Transformers: 5.16.0.dev0
PEFT        : 0.20.0
GPU         : NVIDIA A100-SXM4-80GB
VRAM        : 79.3 GB
dtype       : torch.bfloat16
train: (5073, 8)
test : (5074, 7)
dev  : (4413, 12)

answer
a    0.248965
b    0.249359
c    0.258427
d    0.243249
Name: proportion, dtype: float64


## 3. 이미지 경로 및 로컬 SSD 복사

In [ ]:
LOCAL_DATA_ROOT = Path("/content/vqa_data")

if COPY_IMAGES_TO_LOCAL:
    for sub in ["train", "test", "dev"]:
        src = ROOT / sub
        dst = LOCAL_DATA_ROOT / sub

        if not src.exists():
            continue

        dst.parent.mkdir(parents=True, exist_ok=True)

        if not dst.exists():
            print(f"copy: {src} -> {dst}")
            shutil.copytree(src, dst)
        else:
            print("already exists:", dst)

DATA_ROOT = (
    LOCAL_DATA_ROOT
    if (LOCAL_DATA_ROOT / "train").exists()
    else ROOT
)

print("DATA_ROOT:", DATA_ROOT)


def resolve_image_path(p):
    p = Path(str(p))

    candidates = []

    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.extend([
            DATA_ROOT / p,
            ROOT / p,
            DATA_ROOT / "train" / p.name,
            DATA_ROOT / "test" / p.name,
            DATA_ROOT / "dev" / p.name,
            ROOT / "train" / p.name,
            ROOT / "test" / p.name,
            ROOT / "dev" / p.name,
        ])

    for c in candidates:
        if c.exists():
            return c

    raise FileNotFoundError(f"image not found: {p}")


for df_name, df in [("train", train_df), ("test", test_df)]:
    bad = []
    for p in df["path"].head(100):
        try:
            resolve_image_path(p)
        except FileNotFoundError:
            bad.append(str(p))

    print(df_name, "missing among first 100:", len(bad), bad[:3])

# 함수 정의/경로 셀 누락 방지용 확인
print("first train image:", resolve_image_path(train_df.iloc[0]["path"]))

copy: /content/drive/MyDrive/ssafy-16-1-ai-8-28/train -> /content/vqa_data/train
copy: /content/drive/MyDrive/ssafy-16-1-ai-8-28/test -> /content/vqa_data/test
copy: /content/drive/MyDrive/ssafy-16-1-ai-8-28/dev -> /content/vqa_data/dev
DATA_ROOT: /content/vqa_data
train missing among first 100: 0 []
test missing among first 100: 0 []
first train image: /content/vqa_data/train/train_0001.jpg


## 4. Processor / Qwen3.5 prompt 형식

Qwen3.5-27B의 현재 vision processor는 patch size 16, merge size 2이므로
대략 한 merged visual token이 `32 x 32` 픽셀 영역에 대응하도록 pixel budget을 잡습니다.

기존 Qwen3-VL-8B와의 비교를 위해 최대 visual token은 우선 1024로 유지합니다.

In [ ]:
# Qwen3.5: patch_size=16, merge_size=2 -> 32x32 pixel/token 근사
VISUAL_TOKEN_SIDE = 32

MIN_PIXELS = MIN_VISUAL_TOKENS * VISUAL_TOKEN_SIDE * VISUAL_TOKEN_SIDE
MAX_PIXELS = MAX_VISUAL_TOKENS * VISUAL_TOKEN_SIDE * VISUAL_TOKEN_SIDE

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

processor.tokenizer.padding_side = "right"

print("processor:", type(processor).__name__)
print("min_pixels:", MIN_PIXELS)
print("max_pixels:", MAX_PIXELS)


SYSTEM_INSTRUCT = (
    "You are a highly accurate visual multiple-choice question answering model. "
    "Use the image and the answer choices. "
    "Return exactly one lowercase letter: a, b, c, or d."
)


def build_mc_prompt(question, options):
    return (
        f"질문: {question}\n"
        f"(a) {options[0]}\n"
        f"(b) {options[1]}\n"
        f"(c) {options[2]}\n"
        f"(d) {options[3]}\n\n"
        "이미지와 선택지를 함께 판단하세요. "
        "최종 답은 a, b, c, d 중 한 글자만 출력하세요."
    )


def make_messages(image, question, options, gold=None):
    msgs = [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": SYSTEM_INSTRUCT}
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": build_mc_prompt(question, options)},
            ],
        },
    ]

    if gold is not None:
        msgs.append({
            "role": "assistant",
            "content": [
                {"type": "text", "text": gold}
            ],
        })

    return msgs


def apply_template(messages, add_generation_prompt):
    # Qwen3.5는 non-thinking을 명시하면 generation prompt 뒤에
    # <think>\n\n</think>\n\n 를 넣고 바로 final answer를 예측하게 됩니다.
    return processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        enable_thinking=False,
    )


# a/b/c/d가 각각 single token인지 새 tokenizer에서 반드시 다시 확인
CHOICE_TOKEN_IDS = []

for c in CHOICES:
    ids = processor.tokenizer.encode(
        c,
        add_special_tokens=False,
    )
    print(c, ids)

    assert len(ids) == 1, (
        f"{c!r}가 single token이 아닙니다: {ids}. "
        "이 경우 next-token scoring 코드를 수정해야 합니다."
    )

    CHOICE_TOKEN_IDS.append(ids[0])

CHOICE_TOKEN_IDS = torch.tensor(
    CHOICE_TOKEN_IDS,
    dtype=torch.long,
)

# template sanity check
sample_row = train_df.iloc[0]
sample_image = Image.open(resolve_image_path(sample_row["path"])).convert("RGB")
sample_options = [str(sample_row[c]) for c in CHOICES]

sample_eval_messages = make_messages(
    sample_image,
    str(sample_row["question"]),
    sample_options,
    gold=None,
)

sample_eval_text = apply_template(
    sample_eval_messages,
    add_generation_prompt=True,
)

print()
print("=== generation prompt tail ===")
print(repr(sample_eval_text[-180:]))
print("================================")

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

processor: Qwen3VLProcessor
min_pixels: 262144
max_pixels: 1048576
a [64]
b [65]
c [66]
d [67]

=== generation prompt tail ===
'ad|><|vision_end|>질문: 사진 속 흰색 텀블러의 재질은 무엇인가요?\n(a) 유리\n(b) 금속\n(c) 종이\n(d) 플라스틱\n\n이미지와 선택지를 함께 판단하세요. 최종 답은 a, b, c, d 중 한 글자만 출력하세요.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


## 5. Dataset / AnswerOnly collator

In [ ]:
class TrainVQADataset(Dataset):
    def __init__(self, df, augment_options=True):
        self.df = df.reset_index(drop=True)
        self.augment_options = augment_options

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(
            resolve_image_path(row["path"])
        ).convert("RGB")

        options = [
            str(row[c])
            for c in CHOICES
        ]

        gold_orig = CHOICES.index(
            str(row["answer"]).strip().lower()
        )

        if self.augment_options:
            # displayed position -> original option
            perm = np.random.permutation(4).tolist()
        else:
            perm = [0, 1, 2, 3]

        options_perm = [
            options[j]
            for j in perm
        ]

        gold_display = perm.index(gold_orig)
        gold = CHOICES[gold_display]

        messages = make_messages(
            image,
            str(row["question"]),
            options_perm,
            gold=gold,
        )

        return {
            "messages": messages,
            "image": image,
            "gold": gold,
        }


@dataclass
class AnswerOnlyCollator:
    processor: Any

    def __call__(self, batch):
        texts = []
        images = []
        golds = []

        for x in batch:
            texts.append(
                apply_template(
                    x["messages"],
                    add_generation_prompt=False,
                )
            )
            images.append(x["image"])
            golds.append(x["gold"])

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        # 프롬프트/think 토큰은 전부 무시하고
        # assistant의 정답 letter 한 토큰에만 LM loss를 적용.
        labels = torch.full_like(
            enc["input_ids"],
            -100,
        )

        for b, gold in enumerate(golds):
            gold_id = int(
                self.processor.tokenizer.encode(
                    gold,
                    add_special_tokens=False,
                )[0]
            )

            valid = enc["attention_mask"][b].bool()

            positions = torch.where(
                (enc["input_ids"][b] == gold_id)
                &
                valid
            )[0]

            if len(positions) == 0:
                raise RuntimeError(
                    f"gold token not found: {gold}"
                )

            # assistant 응답의 마지막 a/b/c/d
            answer_pos = int(positions[-1])
            labels[b, answer_pos] = gold_id

        enc["labels"] = labels

        return enc

## 6. 평가 / TTA 함수

In [ ]:
class EvalVQADataset(Dataset):
    def __init__(
        self,
        df,
        perm=(0, 1, 2, 3),
        with_gold=True,
    ):
        self.df = df.reset_index(drop=True)
        self.perm = tuple(perm)
        self.with_gold = with_gold

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(
            resolve_image_path(row["path"])
        ).convert("RGB")

        orig_options = [
            str(row[c])
            for c in CHOICES
        ]

        shown_options = [
            orig_options[j]
            for j in self.perm
        ]

        messages = make_messages(
            image,
            str(row["question"]),
            shown_options,
            gold=None,
        )

        out = {
            "messages": messages,
            "image": image,
            "id": str(row["id"]),
        }

        if self.with_gold:
            out["gold"] = (
                str(row["answer"])
                .strip()
                .lower()
            )

        return out


@dataclass
class EvalCollator:
    processor: Any
    with_gold: bool = True

    def __call__(self, batch):
        texts = []
        images = []

        for x in batch:
            texts.append(
                apply_template(
                    x["messages"],
                    add_generation_prompt=True,
                )
            )
            images.append(x["image"])

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        meta = {
            "id": [x["id"] for x in batch]
        }

        if self.with_gold:
            meta["gold"] = [
                x["gold"]
                for x in batch
            ]

        return enc, meta


def last_nonpad_index(attention_mask):
    rev = torch.flip(
        attention_mask,
        dims=[1],
    )

    offset = rev.float().argmax(dim=1)

    return (
        attention_mask.shape[1]
        - 1
        - offset
    )


PERM_BANK = [
    (0, 1, 2, 3),
    (1, 3, 0, 2),
    (2, 0, 3, 1),
    (3, 2, 1, 0),
]


def get_model_device(model):
    return next(model.parameters()).device


@torch.inference_mode()
def predict_choice_probs(
    model,
    df,
    perm=(0, 1, 2, 3),
    with_gold=False,
    desc="predict",
):
    device = get_model_device(model)

    ds = EvalVQADataset(
        df,
        perm=perm,
        with_gold=with_gold,
    )

    dl = DataLoader(
        ds,
        batch_size=INFER_BATCH,
        shuffle=False,
        collate_fn=EvalCollator(
            processor,
            with_gold=with_gold,
        ),
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    model.eval()

    all_probs = []
    ids = []
    golds = []

    choice_ids = CHOICE_TOKEN_IDS.to(device)

    for enc, meta in tqdm(
        dl,
        desc=desc,
    ):
        enc = {
            k: (
                v.to(
                    device,
                    non_blocking=True,
                )
                if torch.is_tensor(v)
                else v
            )
            for k, v in enc.items()
        }

        with torch.autocast(
            "cuda",
            dtype=DTYPE,
        ):
            out = model(
                **enc,
                use_cache=False,
            )

        logits_device = out.logits.device

        idx = last_nonpad_index(
            enc["attention_mask"]
        ).to(logits_device)

        bidx = torch.arange(
            out.logits.shape[0],
            device=logits_device,
        )

        choice_ids_local = choice_ids.to(
            logits_device
        )

        next_logits = (
            out.logits[bidx, idx]
            [:, choice_ids_local]
        )

        probs = torch.softmax(
            next_logits.float(),
            dim=-1,
        )

        all_probs.append(
            probs.cpu()
        )

        ids.extend(meta["id"])

        if with_gold:
            golds.extend(meta["gold"])

        del out, next_logits, probs, enc

    return (
        torch.cat(all_probs).numpy(),
        ids,
        golds,
    )


def predict_tta(
    model,
    df,
    n_perms=4,
    with_gold=False,
    desc_prefix="TTA",
    cache_dir=None,
    cache_tag=None,
):
    perms = PERM_BANK[:n_perms]

    original_prob_runs = []
    ids_ref = None
    gold_ref = None

    for pi, perm in enumerate(
        perms,
        start=1,
    ):
        cache_file = None

        if cache_dir is not None:
            cache_dir = Path(cache_dir)
            cache_dir.mkdir(
                parents=True,
                exist_ok=True,
            )

            cache_file = (
                cache_dir
                /
                f"{cache_tag}_perm{pi}.npz"
            )

        if (
            cache_file is not None
            and
            cache_file.exists()
        ):
            print(
                f"[cache] load {pi}/{n_perms}:",
                cache_file,
            )

            z = np.load(
                cache_file,
                allow_pickle=True,
            )

            original_probs = z[
                "original_probs"
            ]

            ids = z["ids"].tolist()

            golds = (
                z["golds"].tolist()
                if "golds" in z.files
                else []
            )

        else:
            shown_probs, ids, golds = (
                predict_choice_probs(
                    model,
                    df,
                    perm=perm,
                    with_gold=with_gold,
                    desc=(
                        f"{desc_prefix} "
                        f"{pi}/{n_perms}"
                    ),
                )
            )

            original_probs = np.zeros_like(
                shown_probs
            )

            # displayed j -> original perm[j]
            for shown_j, orig_j in enumerate(
                perm
            ):
                original_probs[
                    :,
                    orig_j
                ] = shown_probs[
                    :,
                    shown_j
                ]

            if cache_file is not None:
                save_dict = {
                    "original_probs": original_probs,
                    "ids": np.array(
                        ids,
                        dtype=object,
                    ),
                }

                if with_gold:
                    save_dict["golds"] = np.array(
                        golds,
                        dtype=object,
                    )

                np.savez_compressed(
                    cache_file,
                    **save_dict,
                )

                print(
                    "saved:",
                    cache_file,
                )

        if ids_ref is None:
            ids_ref = ids

            if with_gold:
                gold_ref = golds

        else:
            assert ids == ids_ref

        original_prob_runs.append(
            original_probs
        )

    runs = np.stack(
        original_prob_runs,
        axis=0,
    )  # [TTA, N, 4]

    avg_probs = runs.mean(axis=0)

    return {
        "runs": runs,
        "avg_probs": avg_probs,
        "ids": ids_ref,
        "golds": gold_ref,
    }


def accuracy_from_probs(
    probs,
    golds,
):
    preds = np.array(CHOICES)[
        probs.argmax(axis=1)
    ]

    golds = np.array(golds)

    return float(
        (preds == golds).mean()
    )

## 7. A100 BF16 → OOM 시 8-bit LoRA model builder

`Qwen3.5-27B`는 9B보다 훨씬 크므로 단순히 BF16을 강제하지 않습니다.

`MODEL_PRECISION="auto"`에서는:

1. BF16 base를 A100에 직접 로드
2. fresh LoRA 부착
3. 실제 train batch로 `forward + backward` preflight
4. CUDA OOM이면 메모리를 해제
5. 8-bit base + `prepare_model_for_kbit_training()` + fresh LoRA로 자동 재시도

Qwen3.5의 full-attention projection과 linear-attention projection, MLP projection을 실제 모델에서 탐지해 LoRA target으로 사용합니다.

In [ ]:
LORA_CANDIDATES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "in_proj_qkv",
    "in_proj_z",
    "in_proj_a",
    "in_proj_b",
    "out_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]


def is_cuda_oom(exc):
    if isinstance(exc, torch.OutOfMemoryError):
        return True

    msg = str(exc).lower()

    return (
        "cuda out of memory" in msg
        or "out of memory" in msg
        or "cublas_status_alloc_failed" in msg
    )


def cleanup_model(model=None):
    if model is not None:
        try:
            model.zero_grad(set_to_none=True)
        except Exception:
            pass

        del model

    gc.collect()
    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass


def gpu_memory_report(prefix="GPU"):
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3

    print(
        f"{prefix} | "
        f"allocated={allocated:.2f} GB, "
        f"reserved={reserved:.2f} GB, "
        f"peak={peak:.2f} GB"
    )


def detect_lora_targets(model):
    present = {}

    for name, module in model.named_modules():
        leaf = name.rsplit(".", 1)[-1]

        if (
            leaf in LORA_CANDIDATES
            and isinstance(module, torch.nn.Linear)
        ):
            present[leaf] = present.get(leaf, 0) + 1

    targets = [
        name
        for name in LORA_CANDIDATES
        if name in present
    ]

    print("LoRA targets:", targets)
    print("module counts:", present)

    assert len(targets) >= 7, (
        "예상한 Qwen3.5 projection module을 충분히 찾지 못했습니다. "
        "Transformers/model architecture를 확인하세요."
    )

    return targets


def build_fresh_lora_model(precision):
    precision = precision.lower()

    assert precision in {"bf16", "8bit"}, precision

    print()
    print("=" * 70)
    print(f"Loading fresh base: {MODEL_ID} | precision={precision}")
    print("=" * 70)

    torch.cuda.reset_peak_memory_stats()

    common_kwargs = {
        "low_cpu_mem_usage": True,
        "attn_implementation": "sdpa",
        "device_map": {"": 0},
    }

    if precision == "bf16":
        common_kwargs["torch_dtype"] = torch.bfloat16

        model = AutoModelForMultimodalLM.from_pretrained(
            MODEL_ID,
            **common_kwargs,
        )

    else:
        quant_config = BitsAndBytesConfig(
            load_in_8bit=True,
        )

        common_kwargs["torch_dtype"] = torch.bfloat16
        common_kwargs["quantization_config"] = quant_config

        model = AutoModelForMultimodalLM.from_pretrained(
            MODEL_ID,
            **common_kwargs,
        )

        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={
                "use_reentrant": False
            },
        )

    if precision == "bf16":
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={
                    "use_reentrant": False
                }
            )

        if hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()

    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False

    targets = detect_lora_targets(model)

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=targets,
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(
        model,
        lora_config,
    )

    model.print_trainable_parameters()

    gpu_memory_report(
        f"after {precision} load"
    )

    return model

## 8. 공통 학습 함수

In [ ]:
def make_train_loader(
    df,
    shuffle=True,
):
    dataset = TrainVQADataset(
        df,
        augment_options=True,
    )

    loader = DataLoader(
        dataset,
        batch_size=TRAIN_BATCH,
        shuffle=shuffle,
        collate_fn=AnswerOnlyCollator(
            processor
        ),
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    return dataset, loader


def preflight_training_step(
    model,
    train_df_local,
    precision,
):
    # 실제 VQA batch 1개로 forward + backward까지 수행해
    # 현재 precision/visual token 설정이 A100에서 학습 가능한지 확인.
    # optimizer.step()은 하지 않으므로 weight는 바뀌지 않습니다.
    print()
    print(
        f"Preflight forward+backward: precision={precision}"
    )

    _, loader = make_train_loader(
        train_df_local.head(
            min(32, len(train_df_local))
        ),
        shuffle=False,
    )

    batch = next(iter(loader))

    print("batch load success ✅")

    device = get_model_device(model)

    for k, v in batch.items():
        if torch.is_tensor(v):
            print(
                k,
                tuple(v.shape),
                v.dtype,
            )

    batch = {
        k: (
            v.to(
                device,
                non_blocking=True,
            )
            if torch.is_tensor(v)
            else v
        )
        for k, v in batch.items()
    }

    model.train()
    model.zero_grad(
        set_to_none=True
    )

    torch.cuda.reset_peak_memory_stats()

    with torch.autocast(
        "cuda",
        dtype=torch.bfloat16,
    ):
        out = model(
            **batch,
            use_cache=False,
        )

        loss = out.loss

    if not torch.isfinite(loss):
        raise RuntimeError(
            f"preflight non-finite loss: {loss}"
        )

    loss.backward()

    print(
        "preflight loss:",
        float(loss.detach()),
    )

    gpu_memory_report(
        f"preflight {precision}"
    )

    model.zero_grad(
        set_to_none=True
    )

    del batch, out, loss
    gc.collect()
    torch.cuda.empty_cache()

    print("preflight success ✅")


def build_model_with_auto_fallback(
    train_df_local,
    requested_precision="auto",
):
    requested_precision = (
        requested_precision
        .lower()
        .strip()
    )

    assert requested_precision in {
        "auto",
        "bf16",
        "8bit",
    }

    if requested_precision == "auto":
        attempts = [
            "bf16",
            "8bit",
        ]
    else:
        attempts = [
            requested_precision
        ]

    last_exc = None

    for precision in attempts:
        model = None

        try:
            model = build_fresh_lora_model(
                precision
            )

            preflight_training_step(
                model,
                train_df_local,
                precision,
            )

            print()
            print(
                f"✅ ACTIVE_PRECISION = {precision}"
            )

            return (
                model,
                precision,
            )

        except Exception as exc:
            last_exc = exc

            if (
                requested_precision == "auto"
                and precision == "bf16"
                and is_cuda_oom(exc)
            ):
                print()
                print(
                    "⚠️ BF16 preflight에서 CUDA OOM 발생."
                )
                print(
                    "BF16 model을 해제하고 8-bit LoRA로 자동 fallback합니다."
                )

                cleanup_model(model)
                continue

            cleanup_model(model)

            if is_cuda_oom(exc):
                raise RuntimeError(
                    f"{precision}에서도 CUDA OOM이 발생했습니다. "
                    "MAX_VISUAL_TOKENS를 1024 -> 768로 낮춘 뒤 "
                    "런타임을 재시작하고 다시 실행하세요."
                ) from exc

            raise

    raise RuntimeError(
        "model build failed"
    ) from last_exc


def train_epochs(
    model,
    train_df_local,
    epochs,
    lr,
    epoch_save_prefix,
):
    device = get_model_device(model)

    train_dataset, train_loader = (
        make_train_loader(
            train_df_local,
            shuffle=True,
        )
    )

    print("train samples:", len(train_dataset))
    print("train batches:", len(train_loader))

    trainable_params = [
        p
        for p in model.parameters()
        if p.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=lr,
        betas=(0.9, 0.95),
        weight_decay=0.01,
    )

    steps_per_epoch = math.ceil(
        len(train_loader)
        / GRAD_ACCUM
    )

    total_steps = (
        epochs
        * steps_per_epoch
    )

    warmup_steps = int(
        total_steps
        * 0.05
    )

    scheduler = (
        get_cosine_schedule_with_warmup(
            optimizer,
            warmup_steps,
            total_steps,
        )
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    history = []

    for epoch in range(
        1,
        epochs + 1,
    ):
        model.train()

        running = 0.0
        seen = 0

        pbar = tqdm(
            train_loader,
            desc=(
                f"Epoch {epoch}/{epochs} train"
            ),
        )

        for step, batch in enumerate(
            pbar,
            start=1,
        ):
            batch = {
                k: (
                    v.to(
                        device,
                        non_blocking=True,
                    )
                    if torch.is_tensor(v)
                    else v
                )
                for k, v in batch.items()
            }

            try:
                with torch.autocast(
                    "cuda",
                    dtype=torch.bfloat16,
                ):
                    out = model(
                        **batch,
                        use_cache=False,
                    )

                    loss = (
                        out.loss
                        / GRAD_ACCUM
                    )

                if not torch.isfinite(loss):
                    raise RuntimeError(
                        f"non-finite loss: {loss}"
                    )

                loss.backward()

            except Exception as exc:
                if is_cuda_oom(exc):
                    print()
                    gpu_memory_report(
                        "OOM at training"
                    )

                    raise RuntimeError(
                        "학습 중 CUDA OOM이 발생했습니다. "
                        "BF16 설정이었다면 MODEL_PRECISION='8bit'로 재실행하거나, "
                        "8-bit에서도 발생했다면 MAX_VISUAL_TOKENS=768로 낮추세요."
                    ) from exc

                raise

            running += (
                float(loss.detach())
                * GRAD_ACCUM
            )

            seen += 1

            do_step = (
                step % GRAD_ACCUM == 0
                or step == len(train_loader)
            )

            if do_step:
                torch.nn.utils.clip_grad_norm_(
                    trainable_params,
                    1.0,
                )

                optimizer.step()

                optimizer.zero_grad(
                    set_to_none=True
                )

                scheduler.step()

            pbar.set_postfix(
                loss=(
                    f"{running / max(seen, 1):.4f}"
                ),
                lr=(
                    f"{scheduler.get_last_lr()[0]:.2e}"
                ),
            )

            del out, loss, batch

        epoch_dir = ROOT / (
            f"{epoch_save_prefix}_epoch{epoch}"
        )

        model.save_pretrained(
            epoch_dir
        )

        processor.save_pretrained(
            epoch_dir
        )

        epoch_loss = (
            running
            / max(seen, 1)
        )

        history.append({
            "epoch": epoch,
            "loss": epoch_loss,
            "adapter_dir": str(epoch_dir),
        })

        print(
            f"epoch={epoch} "
            f"train_loss={epoch_loss:.6f}"
        )

        print(
            "saved adapter:",
            epoch_dir,
        )

        gpu_memory_report(
            f"after epoch {epoch}"
        )

        gc.collect()
        torch.cuda.empty_cache()

    return history

# 9. 전체 train 5,073개 최종 학습

validation split 없이 `train.csv` 전체를 사용합니다.

먼저 실제 train batch로 `forward + backward` preflight를 수행합니다.

- BF16 성공 → BF16 LoRA로 전체 학습
- BF16 CUDA OOM → 자동으로 8-bit LoRA 재로드 후 재시도
- 8-bit도 OOM → `MAX_VISUAL_TOKENS = 768`로 낮춰 재실행

In [ ]:
full_train_df = (
    train_df
    .copy()
    .reset_index(drop=True)
)

print("FULL TRAIN:", len(full_train_df))
assert len(full_train_df) == len(train_df)

final_model, ACTIVE_PRECISION = (
    build_model_with_auto_fallback(
        full_train_df,
        requested_precision=MODEL_PRECISION,
    )
)

print()
print("=" * 70)
print("FINAL TRAIN START")
print("ACTIVE_PRECISION:", ACTIVE_PRECISION)
print("MAX_VISUAL_TOKENS:", MAX_VISUAL_TOKENS)
print("=" * 70)

full_history = train_epochs(
    final_model,
    full_train_df,
    epochs=FULL_EPOCHS,
    lr=FULL_LR,
    epoch_save_prefix=(
        f"qwen35_27b_FINAL_"
        f"full5073_{ACTIVE_PRECISION}_"
        f"v{MAX_VISUAL_TOKENS}"
    ),
)

FINAL_ADAPTER_DIR = ROOT / (
    f"qwen35_27b_FINAL_"
    f"full5073_{ACTIVE_PRECISION}_"
    f"v{MAX_VISUAL_TOKENS}_"
    f"epoch{FULL_EPOCHS}"
)

FINAL_RUN_CONFIG = ROOT / (
    f"qwen35_27b_FINAL_"
    f"full5073_v{MAX_VISUAL_TOKENS}_"
    f"epoch{FULL_EPOCHS}_run_config.json"
)

final_run_config = {
    "model_id": MODEL_ID,
    "precision": ACTIVE_PRECISION,
    "min_visual_tokens": int(MIN_VISUAL_TOKENS),
    "max_visual_tokens": int(MAX_VISUAL_TOKENS),
    "full_train_size": int(len(full_train_df)),
    "full_epochs": int(FULL_EPOCHS),
    "full_lr": float(FULL_LR),
    "grad_accum": int(GRAD_ACCUM),
    "lora_r": int(LORA_R),
    "lora_alpha": int(LORA_ALPHA),
    "lora_dropout": float(LORA_DROPOUT),
    "tta_perms": int(TTA_PERMS),
    "adapter_dir": str(FINAL_ADAPTER_DIR),
}

with open(
    FINAL_RUN_CONFIG,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        final_run_config,
        f,
        ensure_ascii=False,
        indent=2,
    )

print()
print("FINAL adapter:", FINAL_ADAPTER_DIR)
print("FINAL run config:", FINAL_RUN_CONFIG)

FULL TRAIN: 5073

Loading fresh base: Qwen/Qwen3.5-27B | precision=bf16


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/127k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

LoRA targets: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj_qkv', 'in_proj_z', 'in_proj_a', 'in_proj_b', 'out_proj', 'gate_proj', 'up_proj', 'down_proj']
module counts: {'out_proj': 48, 'in_proj_qkv': 48, 'in_proj_z': 48, 'in_proj_b': 48, 'in_proj_a': 48, 'gate_proj': 64, 'up_proj': 64, 'down_proj': 64, 'q_proj': 16, 'k_proj': 16, 'v_proj': 16, 'o_proj': 16}
trainable params: 116,727,808 || all params: 27,473,456,368 || trainable%: 0.4249
after bf16 load | allocated=51.40 GB, reserved=51.41 GB, peak=51.40 GB

Preflight forward+backward: precision=bf16
batch load success ✅
input_ids (1, 780) torch.int64
attention_mask (1, 780) torch.int64
mm_token_type_ids (1, 780) torch.int64
pixel_values (2640, 1536) torch.float32
image_grid_thw (1, 3) torch.int64
labels (1, 780) torch.int64
preflight loss: 1.7639100551605225
preflight bf16 | allocated=52.30 GB, reserved=55.30 GB, peak=54.45 GB
preflight success ✅

✅ ACTIVE_PRECISION = bf16

FINAL TRAIN START
ACTIVE_PRECISION: bf16
MAX_VISUAL_TOKE

Epoch 1/1 train:   0%|          | 0/5073 [00:00<?, ?it/s]

epoch=1 train_loss=0.198522
saved adapter: /content/drive/MyDrive/ssafy-16-1-ai-8-28/qwen35_27b_FINAL_full5073_bf16_v1024_epoch1
after epoch 1 | allocated=52.31 GB, reserved=61.79 GB, peak=57.37 GB

FINAL adapter: /content/drive/MyDrive/ssafy-16-1-ai-8-28/qwen35_27b_FINAL_full5073_bf16_v1024_epoch1
FINAL run config: /content/drive/MyDrive/ssafy-16-1-ai-8-28/qwen35_27b_FINAL_full5073_v1024_epoch1_run_config.json


# 10. Test 5,074개 — TTA=4 → submission

TTA cache 이름에 precision / visual token / epoch을 모두 넣어
설정을 바꿨을 때 오래된 cache가 잘못 재사용되지 않도록 했습니다.

In [ ]:
SUBMISSION_DIR = ROOT / "submission"
SUBMISSION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RUN_TAG = (
    f"qwen35_27b_full5073_"
    f"{ACTIVE_PRECISION}_"
    f"v{MAX_VISUAL_TOKENS}_"
    f"e{FULL_EPOCHS}"
)

TTA_CACHE_DIR = (
    ROOT
    /
    f"tta_cache_{RUN_TAG}"
)

test_tta = predict_tta(
    final_model,
    test_df,
    n_perms=TTA_PERMS,
    with_gold=False,
    desc_prefix="FINAL TEST",
    cache_dir=TTA_CACHE_DIR,
    cache_tag=RUN_TAG,
)

avg_probs = test_tta["avg_probs"]

preds = np.array(CHOICES)[
    avg_probs.argmax(axis=1)
]

submission = pd.DataFrame({
    "id": test_df["id"].values,
    "answer": preds,
})

if sample_df is not None:
    assert set(submission["id"]) == set(sample_df["id"])

    submission = (
        sample_df[["id"]]
        .merge(
            submission,
            on="id",
            how="left",
        )
    )

assert submission["answer"].notna().all()
assert set(submission["answer"].unique()).issubset(set(CHOICES))

OUT = (
    SUBMISSION_DIR
    /
    f"submission_{RUN_TAG}_TTA{TTA_PERMS}.csv"
)

PROB_OUT = (
    SUBMISSION_DIR
    /
    f"{RUN_TAG}_TTA{TTA_PERMS}_probs.npy"
)

submission.to_csv(
    OUT,
    index=False,
)

np.save(
    PROB_OUT,
    avg_probs,
)

print()
print(submission.head())

print()
print(
    submission["answer"]
    .value_counts(normalize=True)
    .sort_index()
)

print()
print("submission:", OUT)
print("probabilities:", PROB_OUT)
print("TTA cache:", TTA_CACHE_DIR)

FINAL TEST 1/4:   0%|          | 0/5074 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/ssafy-16-1-ai-8-28/tta_cache_qwen35_27b_full5073_bf16_v1024_e1/qwen35_27b_full5073_bf16_v1024_e1_perm1.npz


FINAL TEST 2/4:   0%|          | 0/5074 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/ssafy-16-1-ai-8-28/tta_cache_qwen35_27b_full5073_bf16_v1024_e1/qwen35_27b_full5073_bf16_v1024_e1_perm2.npz


FINAL TEST 3/4:   0%|          | 0/5074 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/ssafy-16-1-ai-8-28/tta_cache_qwen35_27b_full5073_bf16_v1024_e1/qwen35_27b_full5073_bf16_v1024_e1_perm3.npz


FINAL TEST 4/4:   0%|          | 0/5074 [00:00<?, ?it/s]

saved: /content/drive/MyDrive/ssafy-16-1-ai-8-28/tta_cache_qwen35_27b_full5073_bf16_v1024_e1/qwen35_27b_full5073_bf16_v1024_e1_perm4.npz

              id answer
0  test_0001.jpg      d
1  test_0002.jpg      b
2  test_0003.jpg      c
3  test_0004.jpg      a
4  test_0005.jpg      b

answer
a    0.249704
b    0.251084
c    0.249901
d    0.249310
Name: proportion, dtype: float64

submission: /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/submission_qwen35_27b_full5073_bf16_v1024_e1_TTA4.csv
probabilities: /content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/qwen35_27b_full5073_bf16_v1024_e1_TTA4_probs.npy
TTA cache: /content/drive/MyDrive/ssafy-16-1-ai-8-28/tta_cache_qwen35_27b_full5073_bf16_v1024_e1


# 11. 학습 완료 후 / TTA 도중 런타임이 끊긴 경우

정상적으로 한 번에 끝났다면 아래 복구 셀은 실행하지 않습니다.

학습 adapter가 Drive에 저장된 뒤 test TTA 중 끊겼다면:

1. 설치 셀 실행 후 런타임 재시작
2. Drive mount부터 평가/TTA 함수 셀까지 다시 실행
3. `RECOVER_FINAL_FOR_TEST = True`
4. 아래 복구 load 셀 실행
5. 복구 TTA 셀 실행

완료된 TTA permutation은 Drive cache에서 다시 계산하지 않습니다.

In [ ]:
RECOVER_FINAL_FOR_TEST = False

RECOVERY_MAX_VISUAL_TOKENS = MAX_VISUAL_TOKENS
RECOVERY_EPOCHS = FULL_EPOCHS

if RECOVER_FINAL_FOR_TEST:
    FINAL_RUN_CONFIG = ROOT / (
        f"qwen35_27b_FINAL_"
        f"full5073_v{RECOVERY_MAX_VISUAL_TOKENS}_"
        f"epoch{RECOVERY_EPOCHS}_run_config.json"
    )

    assert FINAL_RUN_CONFIG.exists(), (
        f"run config not found: {FINAL_RUN_CONFIG}"
    )

    with open(
        FINAL_RUN_CONFIG,
        "r",
        encoding="utf-8",
    ) as f:
        recovery_cfg = json.load(f)

    RECOVERY_PRECISION = recovery_cfg["precision"]
    trained_max_visual = int(
        recovery_cfg["max_visual_tokens"]
    )
    FINAL_ADAPTER_DIR = Path(
        recovery_cfg["adapter_dir"]
    )

    assert trained_max_visual == MAX_VISUAL_TOKENS, (
        "현재 MAX_VISUAL_TOKENS와 학습 당시 값이 다릅니다. "
        f"현재={MAX_VISUAL_TOKENS}, 학습={trained_max_visual}. "
        "상단 설정을 학습 당시 값으로 바꾸고 processor 셀부터 다시 실행하세요."
    )

    assert FINAL_ADAPTER_DIR.exists(), (
        f"adapter not found: {FINAL_ADAPTER_DIR}"
    )

    gc.collect()
    torch.cuda.empty_cache()

    print("Recovery precision:", RECOVERY_PRECISION)
    print("Recovery adapter:", FINAL_ADAPTER_DIR)

    if RECOVERY_PRECISION == "bf16":
        recovery_base = (
            AutoModelForMultimodalLM
            .from_pretrained(
                MODEL_ID,
                torch_dtype=torch.bfloat16,
                attn_implementation="sdpa",
                low_cpu_mem_usage=True,
                device_map={"": 0},
            )
        )

    elif RECOVERY_PRECISION == "8bit":
        recovery_quant = BitsAndBytesConfig(
            load_in_8bit=True,
        )

        recovery_base = (
            AutoModelForMultimodalLM
            .from_pretrained(
                MODEL_ID,
                torch_dtype=torch.bfloat16,
                quantization_config=recovery_quant,
                attn_implementation="sdpa",
                low_cpu_mem_usage=True,
                device_map={"": 0},
            )
        )

    else:
        raise ValueError(RECOVERY_PRECISION)

    recovery_model = (
        PeftModel
        .from_pretrained(
            recovery_base,
            FINAL_ADAPTER_DIR,
            is_trainable=False,
        )
    )

    recovery_model.eval()
    ACTIVE_PRECISION = RECOVERY_PRECISION

    print("recovery model loaded ✅")

else:
    print(
        "복구가 필요할 때만 "
        "RECOVER_FINAL_FOR_TEST=True"
    )

In [ ]:
if RECOVER_FINAL_FOR_TEST:
    SUBMISSION_DIR = ROOT / "submission"
    SUBMISSION_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    RUN_TAG = (
        f"qwen35_27b_full5073_"
        f"{ACTIVE_PRECISION}_"
        f"v{MAX_VISUAL_TOKENS}_"
        f"e{FULL_EPOCHS}"
    )

    TTA_CACHE_DIR = (
        ROOT
        /
        f"tta_cache_{RUN_TAG}"
    )

    recovered_test_tta = predict_tta(
        recovery_model,
        test_df,
        n_perms=TTA_PERMS,
        with_gold=False,
        desc_prefix="RECOVER TEST",
        cache_dir=TTA_CACHE_DIR,
        cache_tag=RUN_TAG,
    )

    avg_probs = recovered_test_tta["avg_probs"]

    preds = np.array(CHOICES)[
        avg_probs.argmax(axis=1)
    ]

    submission = pd.DataFrame({
        "id": test_df["id"].values,
        "answer": preds,
    })

    if sample_df is not None:
        submission = (
            sample_df[["id"]]
            .merge(
                submission,
                on="id",
                how="left",
            )
        )

    OUT = (
        SUBMISSION_DIR
        /
        f"submission_{RUN_TAG}_TTA{TTA_PERMS}.csv"
    )

    PROB_OUT = (
        SUBMISSION_DIR
        /
        f"{RUN_TAG}_TTA{TTA_PERMS}_probs.npy"
    )

    submission.to_csv(
        OUT,
        index=False,
    )

    np.save(
        PROB_OUT,
        avg_probs,
    )

    print("submission:", OUT)
    print("probabilities:", PROB_OUT)
    print("TTA cache:", TTA_CACHE_DIR)

## 실행 방법

처음에는:

```text
패키지 설치
↓
런타임 재시작
↓
Drive mount부터 마지막 일반 실행 셀까지 순서대로 실행
```

하면 됩니다.

가장 좋은 시작 로그:

```text
Loading fresh base: Qwen/Qwen3.5-27B | precision=bf16
preflight success ✅
ACTIVE_PRECISION = bf16
```

BF16이 OOM이면 자동으로 8-bit로 내려갑니다.

8-bit에서도 OOM이면:

```python
MAX_VISUAL_TOKENS = 768
```

로 바꾸고 런타임을 재시작한 뒤 다시 실행하세요.

최종 제출 파일은:

```text
/content/drive/MyDrive/ssafy-16-1-ai-8-28/submission/
```

아래 생성됩니다.